# Appendix A — Least squares, derived

Referenced from Chapters 1.4, 2.1, 3.1, 3.2 and 3.5. This appendix proves the three facts the tutorial
uses: the OLS solution, its equivalence with the minimum-variance hedge ratio, and the standard error.

## A.1 The model and the objective

Observations $t = 1..n$ of an exposure change $y_t = \Delta S_t$ and a hedge-instrument change
$x_t = \Delta F_t$. Fit $y_t = \alpha + \beta x_t + \varepsilon_t$ by choosing $(\alpha, \beta)$ to
minimise the sum of squared residuals

$$
Q(\alpha,\beta) = \sum_{t=1}^{n}\left(y_t - \alpha - \beta x_t\right)^2 .
\tag{A.1}
$$

## A.2 Normal equations

Set both partial derivatives to zero:

$$
\frac{\partial Q}{\partial \alpha} = -2\sum_t (y_t - \alpha - \beta x_t) = 0
\quad\Rightarrow\quad \alpha = \bar y - \beta \bar x
\tag{A.2}
$$

$$
\frac{\partial Q}{\partial \beta} = -2\sum_t x_t (y_t - \alpha - \beta x_t) = 0 .
\tag{A.3}
$$

Substitute (A.2) into (A.3) and write deviations $\tilde x_t = x_t - \bar x$, $\tilde y_t = y_t - \bar y$:

$$
\sum_t x_t(\tilde y_t - \beta\tilde x_t) = 0
\;\Rightarrow\;
\hat\beta = \frac{\sum_t \tilde x_t \tilde y_t}{\sum_t \tilde x_t^2}
= \frac{\widehat{\text{Cov}}(x,y)}{\widehat{\text{Var}}(x)} .
\tag{A.4}
$$

(The step from $\sum x_t(\cdot)$ to $\sum \tilde x_t(\cdot)$ uses $\sum_t(\tilde y_t - \beta\tilde x_t) = 0$,
which is (A.2).) Equation (A.4) **is** the minimum-variance hedge ratio (3.8) with sample moments — the
equivalence claimed in Chapter 3.2.

## A.3 Matrix form (used for several instruments, Chapter 3.3)

Stack the regressors into $X$ ($n\times k$, first column of ones) and $y$ ($n\times 1$):

$$
Q(\boldsymbol\beta) = (y - X\boldsymbol\beta)^\top(y - X\boldsymbol\beta),\qquad
\nabla Q = -2X^\top(y - X\boldsymbol\beta) = 0
\;\Rightarrow\;
\hat{\boldsymbol\beta} = (X^\top X)^{-1}X^\top y .
\tag{A.5}
$$

With demeaned data the slope block of (A.5) is $\Sigma_{FF}^{-1}\Sigma_{FS}$ — equation (3.6).

## A.4 Why least squares equals minimum variance

Minimising $\sum_t \varepsilon_t^2$ with an intercept is the same as minimising the *sample variance* of
$y_t - \beta x_t$ (the intercept absorbs the mean, and variance is the mean of squared deviations). So the
OLS slope is, by construction, the $\beta$ that makes the hedged position $S - \beta F$ least variable in
the sample. No distributional assumption is needed for this statement.

## A.5 Standard error of the slope

Assume $\varepsilon_t$ are uncorrelated with mean zero and common variance $\sigma_\varepsilon^2$ (the
"classical" assumptions). Because $\hat\beta = \sum_t w_t y_t$ with $w_t = \tilde x_t/\sum\tilde x^2$,

$$
\text{Var}(\hat\beta) = \sigma_\varepsilon^2 \sum_t w_t^2 = \frac{\sigma_\varepsilon^2}{\sum_t \tilde x_t^2}
= \frac{\sigma_\varepsilon^2}{(n-1)\,\hat\sigma_x^2} \approx \frac{\sigma_\varepsilon^2}{n\,\sigma_x^2} .
\tag{A.6}
$$

Estimate $\sigma_\varepsilon^2$ by $s^2 = \sum\hat\varepsilon_t^2/(n-k)$. Using
$\sigma_\varepsilon^2 = \sigma_y^2(1-\rho^2)$ gives the form quoted as (3.11):
$\text{se}(\hat h) = (\sigma_S/\sigma_F)\sqrt{(1-\rho^2)/n}$.

In matrix form, $\text{Var}(\hat{\boldsymbol\beta}) = s^2 (X^\top X)^{-1}$ — which is what
`gashedge.hedging.ols` computes.

## A.6 $R^2$ and hedge effectiveness

Total sum of squares $= \sum\tilde y_t^2$; residual sum of squares $= \sum\hat\varepsilon_t^2$.
$R^2 = 1 - \text{RSS}/\text{TSS}$ is the fraction of the variance of $\Delta S$ explained by $\Delta F$.
For a single regressor $R^2 = \hat\rho^2$, and $1 - R^2$ is the fraction of variance left in the hedged
position — so **$R^2$ is hedge effectiveness** (Ederington 1979), equation (3.10).

## A.7 When the classical assumptions fail

* **Heteroskedasticity** (gas volatility clusters): $\hat\beta$ stays unbiased but (A.6) is wrong; use
  White/Newey–West standard errors or the block bootstrap of Chapter 3.2.
* **Autocorrelated residuals** (mean-reverting basis): same fix.
* **Non-stationary levels**: never regress prices on prices (Chapter 1.5); regress changes.
* **Errors in the regressor** (stale or noisy $F$ marks): attenuation bias pulls $\hat\beta$ toward zero
  — one reason to use liquid, well-marked hedge instruments.

## A.8 Numerical check

In [1]:
import sys, pathlib
_root = pathlib.Path.cwd().resolve()
while not (_root / "gashedge").exists():
    _root = _root.parent
sys.path.insert(0, str(_root))

from datetime import date
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from gashedge import get_logger
from gashedge.plotting import setup_style
from gashedge.hedging import ols
setup_style()
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
log = get_logger("appendixA")
log.info("Environment ready")

2026-09-13 16:47:20.702 | INFO    | appendixA | Environment ready


In [2]:
rng = np.random.default_rng(0)
n, beta_true = 500, 0.65
x = rng.normal(0, 1.2, n)
y = 0.05 + beta_true * x + rng.normal(0, 0.5, n)
fit = ols(y, x, names=["x"])
manual_beta = np.cov(x, y, ddof=1)[0, 1] / np.var(x, ddof=1)
manual_se = np.sqrt((fit.resid @ fit.resid / (n - 2)) / ((x - x.mean()) ** 2).sum())
log.info("OLS slope %.4f == cov/var %.4f ; se %.4f == manual %.4f ; R2 %.3f vs rho^2 %.3f",
         fit.beta[1], manual_beta, fit.se[1], manual_se, fit.r2, np.corrcoef(x, y)[0, 1] ** 2)
fit.summary().round(4)

2026-09-13 16:47:20.705 | INFO    | appendixA | OLS slope 0.6520 == cov/var 0.6520 ; se 0.0173 == manual 0.0173 ; R2 0.741 vs rho^2 0.741


,coef,std_err,t_stat
const,0.0155,0.0210,0.7365
x,0.6520,0.0173,37.7378


---
◀ [Previous](../module3_risk_analysis/05_estimation_error_and_regimes.ipynb) · [Contents](../00_introduction.ipynb) · [Next ▶](B_delivery_hours_and_strips.ipynb)